# 05 Intersection Engrams

Intersection-defined fear/extinction engrams with context-driven retrieval routing.

## PART 0: Imports and parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / "src"):
    c = str(candidate)
    if c not in sys.path:
        sys.path.insert(0, c)

from src.engram.hopfield import HopfieldNetwork
from src.engram.patterns import generate_sparse_pattern
from src.engram.metrics import pattern_overlap, activity_fraction

N         = 2000
KAPPA     = 0.15
K         = int(KAPPA * N)     # 300
P_OLD     = 20
SEED      = 42
rng       = np.random.default_rng(SEED)

# Vector fractions
F_CS = 0.70    # 1400 CS neurons
F_US = 0.45    # 900 US neurons
F_A  = 0.45    # 900 fear-day preallocated neurons
F_B  = F_US * F_A   # ~0.2025, 405 ext-day neurons
                     # chosen so E[ext engram] = E[fear engram]

# A-B near-orthogonality parameter
# 0.0 = fully orthogonal (phi=0 forced)
# 1.0 = fully independent (phi by chance only)
# 0.15 = A neurons are 85% less likely to appear in B
AB_OVERLAP_FRACTION = 0.15

# Retrieval
MAX_STEPS    = 20
BETA_CONTEXT = 0.05   # persistent contextual drive during retrieval
                      # applied ONLY to context component, not CS

# Extinction sweep
N_EXT_TRIALS = [0, 1, 2, 3, 5, 10, 20]

# Figure output directory
FIG_DIR = Path("notebooks/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Print expected engram sizes
exp_fear   = N * F_CS * F_US * F_A
exp_ext    = N * F_CS * F_B
print(f"Expected fear engram:  {exp_fear:.1f} (K={K})")
print(f"Expected ext engram:   {exp_ext:.1f}  (K={K})")
print(f"F_B = F_US * F_A = {F_B:.4f}")
print("Part 0 complete")

## PART 1: Generate sparse binary vectors

In [ ]:
def make_sparse_vec(N, n_active, rng):
    """Binary vector with exactly n_active neurons set to 1."""
    vec = np.zeros(N)
    vec[rng.choice(N, n_active, replace=False)] = 1.0
    return vec

cs_vec = make_sparse_vec(N, int(F_CS * N), rng)
us_vec = make_sparse_vec(N, int(F_US * N), rng)

# A: random draw
a_vec = make_sparse_vec(N, int(F_A * N), rng)

# B: near-orthogonal to A
# Neurons in A are included in B with probability
# AB_OVERLAP_FRACTION times the baseline rate.
# Neurons not in A are included at the baseline rate F_B.
n_b_target = int(F_B * N)
a_idx      = np.where(a_vec == 1)[0]
non_a_idx  = np.where(a_vec == 0)[0]

# Expected B neurons from A: n_b_target * F_A * AB_OVERLAP_FRACTION
# Expected B neurons from non-A: n_b_target * (1-F_A * AB_OVERLAP_FRACTION * ...)
# Simplest implementation: allocate proportionally
n_from_a     = int(n_b_target * F_A * AB_OVERLAP_FRACTION)
n_from_non_a = n_b_target - n_from_a

b_from_a     = rng.choice(a_idx, n_from_a, replace=False)
b_from_non_a = rng.choice(non_a_idx, n_from_non_a, replace=False)
b_idx = np.concatenate([b_from_a, b_from_non_a])
b_vec = np.zeros(N)
b_vec[b_idx] = 1.0

# Novel context C for reinstatement test
# Draw from neurons not in A and not in B
non_ab_idx = np.where((a_vec + b_vec) == 0)[0]
n_c = int(F_B * N)
c_vec = np.zeros(N)
c_vec[rng.choice(non_ab_idx, min(n_c, len(non_ab_idx)), replace=False)] = 1.0

# Report overlap statistics
ab_overlap = int(np.sum(a_vec * b_vec))
ab_chance  = int(N * F_A * F_B)
print(f"\nVector sizes:")
print(f"  CS: {int(cs_vec.sum())}  US: {int(us_vec.sum())}  "
      f"A: {int(a_vec.sum())}  B: {int(b_vec.sum())}  "
      f"C: {int(c_vec.sum())}")
print(f"\nA-B overlap: {ab_overlap} neurons")
print(f"Expected by chance: {ab_chance} neurons")
print(f"Reduction: {1 - ab_overlap/ab_chance:.1%} less than chance")
print(f"(AB_OVERLAP_FRACTION={AB_OVERLAP_FRACTION})")
print("Part 1 complete")

## PART 2: Define engrams as intersections

In [ ]:
# Fear engram: CS ∩ US ∩ A
fear_engram = cs_vec * us_vec * a_vec
fear_k = int(fear_engram.sum())
print(f"Fear engram: {fear_k} neurons (expected {exp_fear:.0f})")

# Extinction engram: CS ∩ B
ext_engram = cs_vec * b_vec
ext_k = int(ext_engram.sum())
print(f"Ext engram:  {ext_k} neurons (expected {exp_ext:.0f})")

assert fear_k > 0, "Fear engram is empty - check vector fractions"
assert ext_k > 0,  "Ext engram is empty - check vector fractions"

# Shared neurons
shared = fear_engram * ext_engram
phi_actual = float(shared.sum()) / fear_k
print(f"Shared: {int(shared.sum())} neurons (phi={phi_actual:.3f})")
print(f"Shared neurons are in CS ∩ US ∩ A ∩ B")

# ----- Composition table -----
print(f"\n{'Component':<12} {'Size':>6} "
      f"{'In fear':>9} {'%fear':>7} "
      f"{'In ext':>8} {'%ext':>7}")
print("-" * 56)

for name, vec in [('CS', cs_vec), ('US', us_vec), ('A', a_vec), ('B', b_vec)]:
    n_vec    = int(vec.sum())
    in_fear  = int(np.sum(fear_engram * vec))
    in_ext   = int(np.sum(ext_engram * vec))
    pct_fear = in_fear / fear_k * 100
    pct_ext  = in_ext  / ext_k  * 100
    print(f"{name:<12} {n_vec:>6} "
          f"{in_fear:>9} {pct_fear:>6.1f}% "
          f"{in_ext:>8} {pct_ext:>6.1f}%")

# Explicit CS and US fraction printout (requested)
cs_frac_fear = np.sum(fear_engram * cs_vec) / fear_k
us_frac_fear = np.sum(fear_engram * us_vec) / fear_k
cs_frac_ext  = np.sum(ext_engram  * cs_vec) / ext_k
us_frac_ext  = np.sum(ext_engram  * us_vec) / ext_k

print(f"\n=== CS and US fractions ===")
print(f"CS fraction of fear engram: {cs_frac_fear:.3f}")
print(f"US fraction of fear engram: {us_frac_fear:.3f}")
print(f"CS fraction of ext engram:  {cs_frac_ext:.3f}")
print(f"US fraction of ext engram:  {us_frac_ext:.3f}")
print(f"(Fear engram = CS∩US∩A so CS and US fractions should both be 1.000)")
print(f"(Ext engram = CS∩B so CS fraction should be 1.000, US fraction depends on US∩B overlap)")

# Fear coverage of US vector (= baseline US completion)
fear_coverage_us = np.sum(fear_engram * us_vec) / int(us_vec.sum())
ext_coverage_us  = np.sum(ext_engram  * us_vec) / int(us_vec.sum())
print(f"\nFear engram covers {fear_coverage_us*100:.1f}% of US vector")
print(f"Ext engram covers  {ext_coverage_us*100:.1f}% of US vector")
print(f"(Fear coverage = baseline US completion at test)")
print("Part 2 complete")

## PART 3: Network initialization and fear storage

In [ ]:
net = HopfieldNetwork(N, KAPPA)
old_patterns = [generate_sparse_pattern(N, KAPPA, rng=rng) for _ in range(P_OLD)]
net.store_patterns(old_patterns)
net.store_pattern_sequential(fear_engram)

print(f"Network: P_OLD={P_OLD} background + fear stored")
print(f"Mean |W|: {np.mean(np.abs(net.W)):.6f}")
print(f"Recurrent field scale (K*mean|W|): {K * np.mean(np.abs(net.W)):.4f}")
print(f"Context drive per neuron (beta): {BETA_CONTEXT:.4f}")
print(f"Cue-to-recurrent ratio: {BETA_CONTEXT / (K * np.mean(np.abs(net.W))):.3f}")
print("Part 3 complete")

## PART 4: Retrieval function and cue definitions

In [ ]:
def run_topk_retrieval(net, init_state, K, max_steps, beta=0.0, ext_input=None):
    """
    Top-k retrieval.
    init_state: binary vector, starting point.
    ext_input:  persistent external drive (context/US vector).
                Applied every step with strength beta.
                Separate from init_state - CS is in init but
                not necessarily in ext_input.
    """
    state = init_state.copy().astype(float)
    ext   = (ext_input.copy() if ext_input is not None else np.zeros(N))
    for step in range(max_steps):
        h = net.W @ state + beta * ext
        topk_idx = np.argpartition(h, -K)[-K:]
        new_state = np.zeros(N)
        new_state[topk_idx] = 1.0
        if np.array_equal(new_state, state):
            break
        state = new_state
    return state, step + 1


def measure_retrieval(final_state, fear_engram, ext_engram, us_vec, fear_k, ext_k):
    return {
        'us_completion': (np.sum(final_state * us_vec) / us_vec.sum()),
        'fear_overlap':  (np.sum(final_state * fear_engram) / fear_k),
        'ext_overlap':   (np.sum(final_state * ext_engram) / ext_k),
        'activity':      activity_fraction(final_state)
    }


# Retrieval cues: CS initializes, context persists
# CS is in init_state but NOT in ext_input
# This prevents CS from biasing both attractors equally
# at every step - only the contextual component drives dynamics
retrieval_cues = {
    'CS only': {
        'init': cs_vec.copy(),
        'ext':  np.zeros(N),    # no persistent drive
        'beta': 0.0
    },
    'CS + A': {
        'init': np.clip(cs_vec + a_vec, 0, 1),
        'ext':  a_vec.copy(),   # A persists, CS does not
        'beta': BETA_CONTEXT
    },
    'CS + B': {
        'init': np.clip(cs_vec + b_vec, 0, 1),
        'ext':  b_vec.copy(),   # B persists, CS does not
        'beta': BETA_CONTEXT
    },
    'CS + C + US': {
        'init': np.clip(cs_vec + c_vec + us_vec, 0, 1),
        'ext':  us_vec.copy(),  # US persists as reinstatement
        'beta': BETA_CONTEXT
    },
}

# Verify fear self-retrieval
fear_self, steps = run_topk_retrieval(net, fear_engram, K, MAX_STEPS, beta=0.0)
print(f"Fear self-retrieval: {pattern_overlap(fear_self, fear_engram):.3f} in {steps} steps")
print("Part 4 complete")

## PART 5: Beta context sweep at n_ext=3

### Beta sweep: minimum persistent drive for context-specific retrieval (n_ext=3 fixed)

In [ ]:
# Build reference network with n_ext=3
net_ref = HopfieldNetwork(N, KAPPA)
for p in old_patterns:
    net_ref.store_pattern_sequential(p)
net_ref.store_pattern_sequential(fear_engram)
for _ in range(3):
    net_ref.store_pattern_sequential(ext_engram)

print(f"\n{'beta':>6} | {'CS->f':>6} {'CS->e':>6} | {'A->f':>6} {'A->e':>6} | {'B->f':>6} {'B->e':>6} | {'US->f':>6} {'US->e':>6} | status")
print("-" * 80)

chosen_beta = None
beta_candidates = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20]

for bt in beta_candidates:
    row = {}
    for name, cue_info in retrieval_cues.items():
        final, _ = run_topk_retrieval(
            net_ref,
            cue_info['init'],
            K,
            MAX_STEPS,
            beta=bt,
            ext_input=cue_info['ext'],
        )
        row[name] = measure_retrieval(final, fear_engram, ext_engram, us_vec, fear_k, ext_k)

    af = row['CS + A']['fear_overlap']
    ae = row['CS + A']['ext_overlap']
    bf = row['CS + B']['fear_overlap']
    be = row['CS + B']['ext_overlap']
    cf = row['CS only']['fear_overlap']
    ce = row['CS only']['ext_overlap']
    uf = row['CS + C + US']['fear_overlap']
    ue = row['CS + C + US']['ext_overlap']

    renewal   = af > 0.5 and af > ae
    ext_ok    = be > 0.5 and be > bf
    reinstate = uf > 0.5 and uf > ue
    status = ('GOOD' if renewal and ext_ok and reinstate
              else 'partial' if (renewal or ext_ok or reinstate)
              else '')

    if status == 'GOOD' and chosen_beta is None:
        chosen_beta = bt

    print(f"{bt:>6.3f} | "
          f"{cf:>6.3f} {ce:>6.3f} | "
          f"{af:>6.3f} {ae:>6.3f} | "
          f"{bf:>6.3f} {be:>6.3f} | "
          f"{uf:>6.3f} {ue:>6.3f} | {status}")

if chosen_beta is not None:
    print(f"\nMinimum GOOD beta: {chosen_beta}")
    BETA_CONTEXT = chosen_beta
else:
    print("\nNo GOOD beta found in sweep range.")
    print("Try increasing AB_OVERLAP_FRACTION or N_EXT_TRIALS.")
    BETA_CONTEXT = 0.05  # fallback

for cue_name in retrieval_cues:
    if cue_name != 'CS only':
        retrieval_cues[cue_name]['beta'] = BETA_CONTEXT

print("Part 5 complete")

## PART 6: Baseline retrieval (n_ext=0)

In [ ]:
results_baseline = {}
for name, cue in retrieval_cues.items():
    final, steps = run_topk_retrieval(
        net,
        cue['init'],
        K,
        MAX_STEPS,
        beta=cue['beta'],
        ext_input=cue['ext'],
    )
    metrics = measure_retrieval(final, fear_engram, ext_engram, us_vec, fear_k, ext_k)
    results_baseline[name] = metrics
    print(f"\n{name}:")
    print(f"  US completion:  {metrics['us_completion']:.3f}")
    print(f"  Fear overlap:   {metrics['fear_overlap']:.3f}")
    print(f"  Ext overlap:    {metrics['ext_overlap']:.3f}")
    print(f"  Activity:       {metrics['activity']:.3f}")
    print(f"  Steps:          {steps}")

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(retrieval_cues))
w = 0.22
labels = list(retrieval_cues.keys())
ax.bar(x - w, [results_baseline[l]['us_completion'] for l in labels], w, color='green', label='US completion')
ax.bar(x,     [results_baseline[l]['fear_overlap']  for l in labels], w, color='red',   label='Fear overlap')
ax.bar(x + w, [results_baseline[l]['ext_overlap']   for l in labels], w, color='blue',  label='Ext overlap')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Overlap')
ax.set_ylim(0, 1)
ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5)
ax.set_title('Baseline retrieval (fear stored, no extinction)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '05_baseline_retrieval.png')
plt.show()
print("Part 6 complete")

## PART 7: Extinction sweep

In [ ]:
all_results = {}

for n_ext in N_EXT_TRIALS:
    net_e = HopfieldNetwork(N, KAPPA)
    for p in old_patterns:
        net_e.store_pattern_sequential(p)
    net_e.store_pattern_sequential(fear_engram)

    for _ in range(n_ext):
        net_e.store_pattern_sequential(ext_engram)

    trial_results = {}
    for name, cue in retrieval_cues.items():
        final, steps = run_topk_retrieval(
            net_e,
            cue['init'],
            K,
            MAX_STEPS,
            beta=cue['beta'],
            ext_input=cue['ext'],
        )
        trial_results[name] = measure_retrieval(final, fear_engram, ext_engram, us_vec, fear_k, ext_k)
        trial_results[name]['steps'] = steps

    all_results[n_ext] = trial_results

    cs = trial_results['CS only']
    a  = trial_results['CS + A']
    b  = trial_results['CS + B']
    u  = trial_results['CS + C + US']

    print(
        f"n_ext={n_ext:2d}: "
        f"CS->f={cs['fear_overlap']:.2f} e={cs['ext_overlap']:.2f} | "
        f"A->f={a['fear_overlap']:.2f} e={a['ext_overlap']:.2f} | "
        f"B->f={b['fear_overlap']:.2f} e={b['ext_overlap']:.2f} | "
        f"US->f={u['fear_overlap']:.2f} e={u['ext_overlap']:.2f} | "
        f"US_comp={u['us_completion']:.2f}"
    )

print("Part 7 complete")

## PART 10: Fear weakening during extinction

### Extinction with fear weakening: anti-Hebbian update

Repeated storage of the extinction pattern deepens the extinction attractor globally, eventually overwhelming fear in all contexts. The missing mechanism is that extinction training should simultaneously weaken the fear attractor - the CS-US prediction error during extinction reduces fear synaptic weights.

This is implemented as a small anti-Hebbian subtraction applied to the fear engram weights during each extinction trial, alongside the Hebbian storage of extinction.

The fear weakening rate eta_fear controls how much the fear basin shallows per extinction trial. This is separate from the extinction storage rate (which remains at the standard 1/Na(1-a) normalization).

In [ ]:
def store_with_fear_weakening(net, ext_engram, fear_engram, eta_fear, N, kappa):
    """
    Store extinction pattern (standard Hebbian) AND
    apply anti-Hebbian weakening to fear engram weights.

    eta_fear: fraction of standard learning rate to
              subtract from fear weights.
              0.0 = no weakening (current behavior)
              1.0 = fully remove one fear storage
    """
    # Standard extinction storage
    net.store_pattern_sequential(ext_engram)

    # Anti-Hebbian fear weakening
    if eta_fear > 0:
        a = kappa
        p_centered = fear_engram - a
        dW = np.outer(p_centered, p_centered)
        np.fill_diagonal(dW, 0)
        dW /= (N * a * (1 - a))
        net.W -= eta_fear * dW


n_ext_values = [1, 2, 3, 5]
eta_values = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

# Store results
sweep_results = {}

print(f"{'n_ext':>6} {'eta':>6} | "
      f"{'A->f':>6} {'A->e':>6} | "
      f"{'B->f':>6} {'B->e':>6} | "
      f"{'US->f':>6} {'US->e':>6} | "
      f"{'renew':>6} {'ext_ok':>6} {'rein':>6} | status")
print("-" * 95)

for n_ext in n_ext_values:
    for eta in eta_values:
        # Build fresh network
        net_w = HopfieldNetwork(N, KAPPA)
        for p in old_patterns:
            net_w.store_pattern_sequential(p)
        net_w.store_pattern_sequential(fear_engram)

        # Extinction trials with weakening
        for _ in range(n_ext):
            store_with_fear_weakening(net_w, ext_engram, fear_engram, eta, N, KAPPA)

        # Retrieval with BETA_CONTEXT
        row = {}
        for name, cue_info in retrieval_cues.items():
            final, _ = run_topk_retrieval(
                net_w,
                cue_info['init'], K, MAX_STEPS,
                beta=BETA_CONTEXT,
                ext_input=cue_info['ext']
            )
            row[name] = measure_retrieval(
                final, fear_engram, ext_engram,
                us_vec, fear_k, ext_k
            )

        af = row['CS + A']['fear_overlap']
        ae = row['CS + A']['ext_overlap']
        bf = row['CS + B']['fear_overlap']
        be = row['CS + B']['ext_overlap']
        uf = row['CS + C + US']['fear_overlap']
        ue = row['CS + C + US']['ext_overlap']

        renewal = af > 0.5 and af > ae
        ext_ok = be > 0.5 and be > bf
        reinstate = uf > 0.5 and uf > ue

        status = (
            'GOOD' if renewal and ext_ok and reinstate
            else 'partial' if sum([renewal, ext_ok, reinstate]) >= 2
            else ''
        )

        sweep_results[(n_ext, eta)] = {
            'renewal': renewal,
            'ext_ok': ext_ok,
            'reinstate': reinstate,
            'status': status,
            'af': af,
            'ae': ae,
            'bf': bf,
            'be': be,
            'uf': uf,
            'ue': ue,
        }

        print(f"{n_ext:>6} {eta:>6.2f} | "
              f"{af:>6.3f} {ae:>6.3f} | "
              f"{bf:>6.3f} {be:>6.3f} | "
              f"{uf:>6.3f} {ue:>6.3f} | "
              f"{str(renewal):>6} {str(ext_ok):>6} "
              f"{str(reinstate):>6} | {status}")

    print()

# Identify GOOD configurations
good_configs = [(n, e) for (n, e), r in sweep_results.items() if r['status'] == 'GOOD']

if good_configs:
    print(f"\nGOOD configurations found: {good_configs}")
    best_n, best_eta = good_configs[0]
    print(f"Using n_ext={best_n}, eta_fear={best_eta}")
else:
    partial = [(n, e) for (n, e), r in sweep_results.items() if r['status'] == 'partial']
    print(f"\nNo GOOD found. Partial configs: {partial}")
    best_n, best_eta = (1, 0.3)
    print(f"Falling back to n_ext={best_n}, eta={best_eta}")

print("Part 10 complete")

## PART 11: Full extinction curves with best parameters

### Full retrieval curves using best (n_ext, eta_fear)

In [ ]:
all_results_w = {}

if Path.cwd().name == 'notebooks' and (Path.cwd().parent / 'notebooks').exists():
    weak_fig_dir = Path.cwd().parent / 'notebooks' / 'figures'
else:
    weak_fig_dir = Path.cwd() / 'notebooks' / 'figures'
weak_fig_dir.mkdir(parents=True, exist_ok=True)

for n_ext in N_EXT_TRIALS:
    net_w2 = HopfieldNetwork(N, KAPPA)
    for p in old_patterns:
        net_w2.store_pattern_sequential(p)
    net_w2.store_pattern_sequential(fear_engram)

    for _ in range(n_ext):
        store_with_fear_weakening(net_w2, ext_engram, fear_engram, best_eta, N, KAPPA)

    trial_results = {}
    for name, cue_info in retrieval_cues.items():
        final, steps = run_topk_retrieval(
            net_w2, cue_info['init'], K, MAX_STEPS,
            beta=BETA_CONTEXT,
            ext_input=cue_info['ext']
        )
        m = measure_retrieval(final, fear_engram, ext_engram, us_vec, fear_k, ext_k)
        m['steps'] = steps
        trial_results[name] = m
    all_results_w[n_ext] = trial_results

cue_order = ['CS only', 'CS + A', 'CS + B', 'CS + C + US']
cue_colors = {
    'CS only': 'black',
    'CS + A': 'tab:orange',
    'CS + B': 'tab:blue',
    'CS + C + US': 'tab:green',
}

# Same 2x2 learning curve plot as Part 8 Figure 1, but with weakening
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, cue_name in zip(axes.ravel(), cue_order):
    us_line = [all_results_w[n][cue_name]['us_completion'] for n in N_EXT_TRIALS]
    fear_line = [all_results_w[n][cue_name]['fear_overlap'] for n in N_EXT_TRIALS]
    ext_line = [all_results_w[n][cue_name]['ext_overlap'] for n in N_EXT_TRIALS]

    ax.plot(N_EXT_TRIALS, us_line, color='green', marker='o', label='US completion')
    ax.plot(N_EXT_TRIALS, fear_line, color='red', marker='o', label='Fear overlap')
    ax.plot(N_EXT_TRIALS, ext_line, color='blue', marker='o', label='Ext overlap')
    ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5)
    ax.set_ylim(0, 1)
    ax.set_title(f"{cue_name} (with fear weakening)")
    ax.set_xlabel('Number of extinction trials stored')
    ax.set_ylabel('Overlap')
    ax.legend()

plt.tight_layout()
plt.savefig(weak_fig_dir / '05_extinction_curves_weakening.png')
plt.show()

# Side-by-side US completion comparison
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for cue_name in cue_order:
    color = cue_colors[cue_name]
    line_no_w = [all_results[n][cue_name]['us_completion'] for n in N_EXT_TRIALS]
    line_w = [all_results_w[n][cue_name]['us_completion'] for n in N_EXT_TRIALS]
    ax_l.plot(N_EXT_TRIALS, line_no_w, marker='o', color=color, label=cue_name)
    ax_r.plot(N_EXT_TRIALS, line_w, marker='o', color=color, label=cue_name)

for ax, title in ((ax_l, 'No fear weakening'), (ax_r, 'With fear weakening')):
    ax.axhline(0.5, linestyle='--', color='gray', alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Number of extinction trials stored')
    ax.set_ylim(0, 1)

ax_l.set_ylabel('US completion (fraction of US neurons reactivated)')
ax_r.legend()
plt.tight_layout()
plt.savefig(weak_fig_dir / '05_us_completion_comparison.png')
plt.show()

print("Part 11 complete")

## PART 12: Print final biological summary

In [ ]:
final_res = all_results_w[best_n]

def _label(fear_val, ext_val):
    return 'FEAR' if fear_val > ext_val else 'EXT'

print("=== FINAL MODEL SUMMARY ===")
print(f"Extinction protocol: n_ext={best_n} trial(s), eta_fear={best_eta} (fear weakening rate)")

fa = final_res['CS + A']['fear_overlap']
ea = final_res['CS + A']['ext_overlap']
ua = final_res['CS + A']['us_completion']

fb = final_res['CS + B']['fear_overlap']
eb = final_res['CS + B']['ext_overlap']
ub = final_res['CS + B']['us_completion']

fu = final_res['CS + C + US']['fear_overlap']
eu = final_res['CS + C + US']['ext_overlap']
uu = final_res['CS + C + US']['us_completion']

fc = final_res['CS only']['fear_overlap']
ec = final_res['CS only']['ext_overlap']
uc = final_res['CS only']['us_completion']

print("\nAfter extinction:")
print(f"  CS+A (fear context):  fear={fa:.3f}, ext={ea:.3f}, US_completion={ua:.3f}  -> {_label(fa, ea)}")
print(f"  CS+B (ext context):   fear={fb:.3f}, ext={eb:.3f}, US_completion={ub:.3f}  -> {_label(fb, eb)}")
print(f"  CS+C+US (reinstate):  fear={fu:.3f}, ext={eu:.3f}, US_completion={uu:.3f}  -> {_label(fu, eu)}")
print(f"  CS only:              fear={fc:.3f}, ext={ec:.3f}, US_completion={uc:.3f}  -> {_label(fc, ec)}")

print("\nKey predictions:")
print(f"1. Fear weakening rate eta={best_eta} maps onto the degree of CS-US prediction error signal during extinction (e.g. dopaminergic omission response).")
print(f"2. Context specificity requires persistent contextual input beta={BETA_CONTEXT} - below this, extinction is context-general.")
print("3. The fear memory is weakened but not erased: it remains retrievable via reinstatement (CS+US).")

print("Part 12 complete")

## PART 8: Plots

In [ ]:
cue_order = ['CS only', 'CS + A', 'CS + B', 'CS + C + US']

# Figure 1 - 2x2 subplots: extinction learning curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, cue_name in zip(axes.ravel(), cue_order):
    us_line = [all_results[n][cue_name]['us_completion'] for n in N_EXT_TRIALS]
    fear_line = [all_results[n][cue_name]['fear_overlap'] for n in N_EXT_TRIALS]
    ext_line = [all_results[n][cue_name]['ext_overlap'] for n in N_EXT_TRIALS]

    ax.plot(N_EXT_TRIALS, us_line, color='green', marker='o', label='US completion')
    ax.plot(N_EXT_TRIALS, fear_line, color='red', marker='o', label='Fear overlap')
    ax.plot(N_EXT_TRIALS, ext_line, color='blue', marker='o', label='Ext overlap')
    ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5)
    ax.set_ylim(0, 1)
    ax.set_title(cue_name)
    ax.set_xlabel('Number of extinction trials stored')
    ax.set_ylabel('Overlap')
    ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / '05_extinction_curves.png')
plt.show()

# Figure 2 - US completion across all conditions vs n_ext
fig, ax = plt.subplots(figsize=(10, 5))
for cue_name in cue_order:
    us_line = [all_results[n][cue_name]['us_completion'] for n in N_EXT_TRIALS]
    ax.plot(N_EXT_TRIALS, us_line, marker='o', label=cue_name)

ax.axhline(0.5, linestyle='--', color='gray', alpha=0.8, label='extinction criterion')
ax.set_title('US completion across retrieval conditions vs extinction trials')
ax.set_xlabel('Number of extinction trials stored')
ax.set_ylabel('US completion (fraction of US neurons reactivated)')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '05_us_completion.png')
plt.show()

# Figure 3 - overlap heatmaps
fear_mat = np.array([[all_results[n][cue]['fear_overlap'] for cue in cue_order] for n in N_EXT_TRIALS])
ext_mat = np.array([[all_results[n][cue]['ext_overlap']  for cue in cue_order] for n in N_EXT_TRIALS])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].imshow(fear_mat, cmap='Reds', vmin=0, vmax=1, aspect='auto')
axes[0].set_title('Fear engram overlap')
axes[0].set_xticks(np.arange(len(cue_order)))
axes[0].set_xticklabels(cue_order, rotation=20, ha='right')
axes[0].set_yticks(np.arange(len(N_EXT_TRIALS)))
axes[0].set_yticklabels(N_EXT_TRIALS)
axes[0].set_ylabel('n_ext')
for i in range(fear_mat.shape[0]):
    for j in range(fear_mat.shape[1]):
        axes[0].text(j, i, f"{fear_mat[i, j]:.2f}", ha='center', va='center', color='black', fontsize=9)
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(ext_mat, cmap='Blues', vmin=0, vmax=1, aspect='auto')
axes[1].set_title('Extinction engram overlap')
axes[1].set_xticks(np.arange(len(cue_order)))
axes[1].set_xticklabels(cue_order, rotation=20, ha='right')
axes[1].set_yticks(np.arange(len(N_EXT_TRIALS)))
axes[1].set_yticklabels(N_EXT_TRIALS)
axes[1].set_ylabel('n_ext')
for i in range(ext_mat.shape[0]):
    for j in range(ext_mat.shape[1]):
        axes[1].text(j, i, f"{ext_mat[i, j]:.2f}", ha='center', va='center', color='black', fontsize=9)
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(FIG_DIR / '05_overlap_heatmaps.png')
plt.show()
print("Part 8 complete")

## PART 9: Summary and context selectivity

In [ ]:
print("=== ENGRAM STRUCTURE ===")
print(f"Fear: {fear_k} neurons | Ext: {ext_k} neurons | Shared: {int(shared.sum())} (phi={phi_actual:.3f})")
print(f"CS fraction of fear engram: {cs_frac_fear:.3f}")
print(f"US fraction of fear engram: {us_frac_fear:.3f}")
print(f"CS fraction of ext engram:  {cs_frac_ext:.3f}")
print(f"US fraction of ext engram:  {us_frac_ext:.3f}")
print(f"Fear engram covers {fear_coverage_us*100:.1f}% of US neurons")

n_eval = 3 if 3 in all_results else max(N_EXT_TRIALS)
res = all_results[n_eval]
renewal_index = res['CS + A']['fear_overlap'] - res['CS + B']['fear_overlap']
extinction_index = res['CS + B']['ext_overlap'] - res['CS + A']['ext_overlap']
reinstatement = res['CS + C + US']['fear_overlap']

pass_list = []
fail_list = []
if renewal_index > 0:
    pass_list.append("renewal_index>0")
else:
    fail_list.append("renewal_index<=0")
if extinction_index > 0:
    pass_list.append("extinction_index>0")
else:
    fail_list.append("extinction_index<=0")
if reinstatement > 0.5:
    pass_list.append("reinstatement>0.5")
else:
    fail_list.append("reinstatement<=0.5")

print("\n=== CONTEXT SELECTIVITY ===")
print(f"At n_ext={n_eval}")
print(f"Renewal index (CS+A fear - CS+B fear):  {renewal_index:.3f}")
print(f"Extinction index (CS+B ext - CS+A ext): {extinction_index:.3f}")
print(f"Reinstatement (CS+C+US fear):           {reinstatement:.3f}")
print(f"PASS: {pass_list}")
print(f"FAIL: {fail_list}")
print("Part 9 complete")

Model predictions:

1. US completion at CS+B drops with extinction trials:
   the extinction context suppresses US-predictive reactivation - behavioral extinction.

2. US completion at CS+A remains high (renewal):
   returning to the fear context reinstates fear responding - the fear memory is not erased.

3. US completion at CS+C+US recovers after extinction (reinstatement):
   US presentation in any context reactivates the fear memory.

4. The minimum BETA_CONTEXT for context-specific routing is a quantitative prediction:
   below this value, persistent contextual input is insufficient to disambiguate the two memories.
   This maps onto the minimum strength of hippocampal-amygdala contextual input required for context-specific extinction.